# Run the approved additional-subset pilot with the resource amendment.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT
REVISION_ROOT = PROJECT_DIR / "revision_outputs"
DATA_DIR = PROJECT_DIR / "data"
PYTHON_EXE = str(RELEASE_CLASSICAL_PYTHON)


In [ ]:
import subprocess,sys,os,time,json,platform
from pathlib import Path
from datetime import datetime,timezone


def python_candidates(explicit=None):
    if explicit:
        return [str(Path(explicit).expanduser().absolute())]
    candidates=[Path(sys.executable)]
    bases={Path(sys.prefix),Path(sys.base_prefix),Path(sys.executable).parent.parent,
           Path('/opt/anaconda3'),Path('/opt/miniconda3'),Path.home()/'miniconda3',Path.home()/'anaconda3'}
    for base in sorted(bases):
        candidates.extend(sorted((base/'envs').glob('*/bin/python')))
    candidates.extend(sorted((Path.home()/'.conda/envs').glob('*/bin/python')))
    registry=Path.home()/'.conda/environments.txt'
    if registry.is_file():
        candidates.extend(Path(line.strip())/'bin/python' for line in registry.read_text().splitlines() if line.strip())
    for base in [Path.cwd(),Path.cwd().parent]:
        candidates.extend([base/'.venv/bin/python',base/'venv/bin/python'])

    unique=[]
    for path in candidates:
        value=str(path.expanduser().absolute())
        if path.is_file() and value not in unique:unique.append(value)
    return unique


def compatible_python(expected,out,explicit=None):
    code="""import json,platform,importlib.metadata as m
v={'python':platform.python_version()}
for key,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('tensorflow','tensorflow'),('keras','keras'),('scipy','scipy'),('joblib','joblib')]:
 try:v[key]=m.version(package)
 except m.PackageNotFoundError:v[key]='NOT_INSTALLED'
print(json.dumps(v))
"""
    reports=[]
    for executable in python_candidates(explicit):
        item={'executable':executable}
        try:
            proc=subprocess.run([executable,'-c',code],capture_output=True,text=True,timeout=20)
            item['metadata_returncode']=proc.returncode
            if proc.returncode!=0:
                item['status']='METADATA_FAILED';item['stderr_tail']=proc.stderr[-1500:];reports.append(item);continue
            versions=json.loads(proc.stdout.strip().splitlines()[-1]);item['versions']=versions
            if any(versions.get(k)!=v for k,v in expected.items()):
                item['status']='VERSIONS_DIFFER';reports.append(item);continue
            print('Eğitim sürümleriyle eşleşen Python kontrol ediliyor:',executable,flush=True)
            probe="import numpy, pandas; import tensorflow as tf; import sklearn; print('TF_PROBE_OK',float(tf.reduce_sum(tf.constant([1.,2.])).numpy()),flush=True)"
            proc=subprocess.run([executable,'-X','faulthandler','-c',probe],capture_output=True,text=True,timeout=60)
            item.update(probe_returncode=proc.returncode,probe_stdout_tail=proc.stdout[-1500:],probe_stderr_tail=proc.stderr[-3000:])
            if proc.returncode==0 and 'TF_PROBE_OK 3.0' in proc.stdout:
                item['status']='READY';reports.append(item);return executable,reports
            item['status']='IMPORT_OR_TENSOR_TEST_FAILED'
        except subprocess.TimeoutExpired:
            item['status']='PROBE_TIMEOUT'
        except (OSError,ValueError) as e:
            item.update(status='PROBE_ERROR',error=str(e))
        reports.append(item)
    return None,reports




import hashlib,fcntl,copy

def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def canonical(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def require(ok,message):
    if not ok:raise ValueError(message)
def write(p,x):
    temp=p.with_suffix('.tmp');temp.write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False));temp.replace(p)
def validate(result,out,job,plan_hash):
    require(result['status']=='PASS_FOLD','Ek veri kontrolü tamamlanmadı: '+result['status'])
    require(result['full_plan_sha256']==plan_hash,'Plan farklı')
    require((result['seed'],result['fold'],result['representation'])==(job['seed'],job['fold'],job['representation']),'İş kimliği farklı')
    require(not result['AE_training_performed'] and not result['test_files_read'],'Beklenmeyen eğitim/veri kullanımı')
    rows=result['primary_results']+result['sensitivity_results']
    keys={(r['representation'],r['method'],r['RUL_window']) for r in rows}
    methods=['DBSCAN','IF','LOF','OC-SVM','kNN']+(['MSE'] if job['representation'].startswith('AE_') else [])
    expected={(job['representation'],m,w) for m in methods for w in [20,30,40]}
    require(len(rows)==len(expected) and keys==expected,'Beş skor hattı / üç RUL penceresi eksik')
    require(len(result['selection_trials'])==18 and all(t['status']=='VALID' for t in result['selection_trials']),'Eksik/geçersiz aday')
    require((result['dataset'],result['preprocessing'])==(job['dataset'],job['preprocessing']),'Veri/ön işleme kimliği farklı')
    require(result['versions']==PLAN['expected_versions'] and result['addendum_sha256']==PLAN['addendum_sha256'],'Sürüm/protokol farklı')
    require(result['PCA_refitted'] is False,'PCA yeniden fit edilmiş')
    for old in job['approved_primary_results']:
        r=next(r for r in result['primary_results'] if r['method']==old['method'])
        require(all(r[k]==v for k,v in old.items()),'Eski10 ana sonucu değişmiş')
    for name,h in result['output_sha256'].items():
        require(Path(name).name==name,'Geçersiz çıktı adı')
        require((out/name).is_file() and digest(out/name)==h,'Kayıtlı çıktı değişmiş: '+name)

def run_pilot():
    root=REVISION_ROOT.expanduser().resolve();data=DATA_DIR.expanduser().resolve()
    ph=canonical(PLAN);out=root/'10B1_FD002_FD003_five_method_pilot'/ph[:20];out.mkdir(parents=True,exist_ok=True)
    summary_path=out/'additional_pilot_summary.json'
    summary={'stage':'10B1_FD002_FD003_five_method_pilot','status':'RUNNING','plan_sha256':ph,'expected_jobs':15,'completed_jobs':0,'PCA_refitted':False,'AE_training_performed':False,'test_files_read':False,'previous_outputs_modified':False,'full_additional_comparison_completed':False,'results':[]}
    with (out/'run.lock').open('a') as lock:
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        try:
            require(hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()==PLAN['worker_sha256'],'Worker değişmiş')

            prior_trials_path=root/'10B_FD002_FD003_five_method_pilot/7007fcbc16eecf6554bd/FD002_global_Raw/attempt_20261101T233312_455204Z/selection_trials.json'
            summary['prior_blocked_candidate']=None
            if prior_trials_path.is_file():
                try:
                    blocked=[t for t in json.loads(prior_trials_path.read_text()) if t.get('status')=='RESOURCE_LIMIT' and t.get('reason')=='DBSCAN would allocate 22112843 neighbor entries; limit=20000000. No candidate will be silently dropped.']
                    if blocked:summary['prior_blocked_candidate']=blocked[-1]
                except (OSError,ValueError):pass
            write(out/'pilot_plan.json',PLAN)
            worker=out/'worker.py';worker.write_text(WORKER_SOURCE)
            exe,reports=compatible_python(PLAN['expected_versions'],out,PYTHON_EXE)
            summary.update(worker_executable=exe,environment_checks=reports)
            require(exe is not None,'Uyumlu Python bulunamadı; environment_checks bölümünü gönderin.')
            for job in PLAN['jobs']:

                for rel,h in job['known_hashes'].items():
                    path=root/rel
                    require(path.is_file() and digest(path)==h,'Kaynak dosya eksik/değişmiş: '+str(path))
                require(digest(data/f"train_{job['dataset']}.txt")==job['train_sha256'],'Ham veri değişmiş')
                jobdir=out/job['job_key'];jobdir.mkdir(exist_ok=True);done=jobdir/'done.json'
                if done.is_file():
                    pointer=json.loads(done.read_text());attempt=jobdir/pointer['attempt']
                    require(attempt.parent==jobdir and attempt.is_dir(),'Geçersiz kayıt yolu')
                    sp=attempt/'fold_completion_summary.json'
                    require(digest(sp)==pointer['summary_sha256'],'Tamamlanmış özet değişmiş')
                    result=json.loads(sp.read_text());validate(result,attempt,job,ph)
                    print('KAYITTAN:',job['job_key'],flush=True)
                else:
                    attempt=jobdir/('attempt_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'));attempt.mkdir()
                    config={**copy.deepcopy(job),'revision_root':str(root),'data_directory':str(data),'output_directory':str(attempt),'expected_versions':PLAN['expected_versions'],'five_method_addendum':PLAN['five_method_addendum'],'addendum_sha256':PLAN['addendum_sha256'],'full_plan_sha256':ph,'approved09_summary':PLAN['approved09_summary']}
                    cp=attempt/'config.json';write(cp,config)
                    print('ÇALIŞIYOR:',job['job_key'],'seed42 fold1',flush=True)
                    with (attempt/'worker.log').open('w') as log:
                        proc=subprocess.Popen([exe,'-u','-X','faulthandler',str(worker),str(cp)],stdout=log,stderr=subprocess.STDOUT)
                        try:
                            last=time.monotonic()
                            while proc.poll() is None:
                                time.sleep(1)
                                if time.monotonic()-last>30:
                                    print(job['representation'],'çalışıyor; ayrıntı:',attempt/'worker.log',flush=True);last=time.monotonic()
                        except BaseException:
                            proc.terminate()
                            try:proc.wait(timeout=10)
                            except subprocess.TimeoutExpired:proc.kill();proc.wait()
                            raise
                    sp=attempt/'fold_completion_summary.json'
                    require(proc.returncode==0 and sp.is_file(),'Alt işlem durdu; worker.log dosyasını inceleyin: '+str(attempt))
                    result=json.loads(sp.read_text())
                    summary['last_job_result']=result
                    validate(result,attempt,job,ph)
                    write(done,{'attempt':attempt.name,'summary_sha256':digest(sp)})
                summary['results'].append(result);summary['completed_jobs']=len(summary['results']);write(summary_path,summary)
                print('TAMAM:',job['job_key'],flush=True)
            require(summary['completed_jobs']==15,'15 iş tamamlanmalı')
            all_rows=[r for j in summary['results'] for r in j['primary_results']+j['sensitivity_results']]
            require(len(all_rows)==243 and len({tuple(r[k] for k in ['dataset','preprocessing','representation','method','RUL_window']) for r in all_rows})==243,'15 iş/81 ana grup kapsamı eksik')
            summary.update(status='PASS_ADDITIONAL_FIVE_METHOD_PILOT',primary_results=[r for j in summary['results'] for r in j['primary_results']],new_candidate_attempts=sum(j['new_candidate_attempts'] for j in summary['results']),limits=['Only seed42 fold1,three original dataset/preprocessing blocks xfive representations; not full450-job CV.','Raw/AE comparisons completed in05B/06C are retained separately, not retrained or dropped.','Same detector grids, scoring and selection. Versioned resource rule: bounded queries above original threshold.','RUL labels are late-life proxies, not annotated anomaly onset.'])
        except KeyboardInterrupt:
            summary.update(status='INTERRUPTED',error='Yeniden Run All ile tamamlanan işler doğrulanıp kullanılabilir.')
        except Exception as e:
            import traceback
            summary.update(status='REVIEW_REQUIRED',error=str(e),traceback=traceback.format_exc());traceback.print_exc()
        finally:
            write(summary_path,summary)
    print('\nDURUM:',summary['status']);print('GÖNDERİLECEK DOSYA:',summary_path)
    return summary,summary_path


In [ ]:
PLAN = load_json('assets/10B1/03_plan.json')
WORKER_SOURCE = load_text('assets/10B1/03_worker_source.py')


In [ ]:
result, summary_path = run_pilot()
